In [1]:
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import polars as pl
import seaborn as sns
from sklearn.model_selection import train_test_split

RANDOM_SEED = 1000

In [2]:
from typing import Optional


def recall_at_k(y_true, y_score, k) -> float:
    top = np.argsort(y_score)[::-1][:k]
    n_true_positives_in_top_k = y_true[top].sum()
    n_true_positives_total = y_true.sum()
    return n_true_positives_in_top_k / n_true_positives_total


def load_and_clean_data(features_path, labels_path=None):
    df = pl.read_parquet(features_path)
    df = df.filter(pl.all_horizontal(pl.col("^.*_valid$")))

    if labels_path is not None:
        df_labels = pl.read_parquet(labels_path)
        df = df.join(df_labels, on="id")
        df = df.filter(pl.col("affinity_kcal_mol") < 0)

    return df


def extract_features_and_target(
    df, feature_cols=None, target_col="affinity_kcal_mol"
) -> tuple[np.ndarray, Optional[np.ndarray], list[str]]:
    if feature_cols is None:
        feature_cols = []

    feature_arrays = []
    feature_names = []

    for col in feature_cols:
        arr = df[col].to_numpy()

        if arr.ndim == 1 and hasattr(arr[0], "__len__"):
            arr = np.stack(arr)

        feature_arrays.append(arr)

        n_features = arr.shape[1] if arr.ndim > 1 else 1
        feature_names.extend([f"{col}_{i}" for i in range(n_features)])

    X = np.hstack(feature_arrays)

    if target_col in df.columns:
        y = df[target_col].to_numpy()
    else:
        y = None

    return X, y, feature_names


def evaluate_enrichment(y_true, y_prob, percents=[0.01, 0.05, 0.10, 0.15]):
    sorted_indices = np.argsort(y_prob)[::-1]
    total_actives = int(y_true.sum())

    results = []

    for pct in percents:
        k = max(1, int(len(y_true) * pct))
        top = sorted_indices[:k]

        prec = y_true[top].mean()
        rec = y_true[top].sum() / total_actives
        n_caught = int(y_true[top].sum())

        print(
            f"top {pct:.0%}: precision={prec:.3f} recall={rec:.3f} "
            f"n_active_caught={n_caught}/{total_actives}"
        )

        results.append({"Top %": pct * 100, "Precision": prec, "Recall": rec})

    return pl.DataFrame(results)


def plot_enrichment(df_results):
    df_melted = df_results.unpivot(
        index="Top %",
        on=["Precision", "Recall"],
        variable_name="Metric",
        value_name="Score",
    )

    plt.figure(figsize=(8, 5))
    sns.set_theme(style="whitegrid")

    ax = sns.lineplot(
        data=df_melted,
        x="Top %",
        y="Score",
        hue="Metric",
        marker="o",
        markersize=8,
        linewidth=2,
    )

    plt.title("Screening Enrichment: Precision vs. Recall", fontsize=14, pad=10)
    plt.xlabel("Top % of Ranked Molecules Screened", fontsize=12)
    plt.ylabel("Score (0.0 to 1.0)", fontsize=12)

    plt.ylim(0, 1.05)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x)}%"))

    plt.tight_layout()
    plt.show()


def prepare_evaluation_data(model, x, y, active_threshold, top_fraction=0.01):
    y_binary = (y <= active_threshold).astype(int)
    p_all_classes = np.asarray(model.predict_proba(x))
    p_active = p_all_classes[:, 1]
    k = max(1, int(len(x) * top_fraction))

    return y_binary, p_active, k

In [3]:
initial_features_file = "data/features/sample_initial_100k_features.parquet"
initial_labels_file = "data/docking_runs/sample_initial_100k.1L83.p2rank.1.parquet"

feature_cols = [
    "atom_pair",
    "autocorr",
    "descriptors",
    "ecfp",
    "functional_groups",
    "morse",
    "rdf",
    "topological_torsion",
    "usrcat",
    "whim",
]

df_clean = load_and_clean_data(initial_features_file, initial_labels_file)
x_initial, y_initial, feature_names = extract_features_and_target(
    df_clean, feature_cols, "affinity_kcal_mol"
)

assert y_initial is not None, "Target column missing; cannot calculate threshold."

active_threshold = np.percentile(y_initial, 1)
y_initial_binary = (y_initial <= active_threshold).astype(int)

x_initial_train, x_initial_test, y_initial_train, y_initial_test = train_test_split(
    x_initial, y_initial_binary, train_size=0.8, random_state=RANDOM_SEED
)

In [4]:
model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=10,
    random_state=RANDOM_SEED,
    deterministic=True,
    force_row_wise=True,
)

model.fit(x_initial_train, y_initial_train, feature_name=feature_names)

[LightGBM] [Info] Number of positive: 761, number of negative: 74219
[LightGBM] [Info] Total Bins 211948
[LightGBM] [Info] Number of data points in the train set: 74980, number of used features: 4616
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.010149 -> initscore=-4.580142
[LightGBM] [Info] Start training from score -4.580142


,learning_rate,0.05
,n_estimators,500
,objective,'binary'
,min_child_samples,10
,random_state,1000
,deterministic,True
,force_row_wise,True
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000


In [5]:
y_initial_test_binary, p_initial_test, k_initial_test = prepare_evaluation_data(
    model=model, x=x_initial_test, y=y_initial_test, active_threshold=active_threshold
)

df_metrics = evaluate_enrichment(y_initial_test, p_initial_test)

top 1%: precision=0.374 recall=0.385 n_active_caught=70/182
top 5%: precision=0.160 recall=0.824 n_active_caught=150/182
top 10%: precision=0.092 recall=0.951 n_active_caught=173/182
top 15%: precision=0.064 recall=0.984 n_active_caught=179/182


/workspaces/lynceus/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [6]:
holdout_features_file = "data/features/sample_holdout_100k_features.parquet"
holdout_labels_file = "data/docking_runs/sample_holdout_100k.1L83.p2rank.1.parquet"

In [7]:
df_holdout_clean = load_and_clean_data(holdout_features_file, holdout_labels_file)
x_holdout, y_holdout, feature_names = extract_features_and_target(
    df_holdout_clean, feature_cols, "affinity_kcal_mol"
)
y_holdout_binary, p_holdout, k_holdout = prepare_evaluation_data(
    model=model, x=x_holdout, y=y_holdout, active_threshold=active_threshold
)
df_holdout_metrics = evaluate_enrichment(y_holdout_binary, p_holdout)

/workspaces/lynceus/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


top 1%: precision=0.406 recall=0.418 n_active_caught=380/909
top 5%: precision=0.166 recall=0.854 n_active_caught=776/909
top 10%: precision=0.093 recall=0.956 n_active_caught=869/909
top 15%: precision=0.064 recall=0.987 n_active_caught=897/909


In [8]:
import glob
import logging

import pyarrow.parquet as pq

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)


def predict_large_parquet(
    file_pattern: str,
    model,
    feature_cols: list[str],
    batch_size: int = 10_000,
    log_interval: int = 100_000,
) -> pl.DataFrame:

    file_paths = glob.glob(file_pattern)
    if not file_paths:
        logging.warning(f"No files found matching pattern '{file_pattern}'")
        return pl.DataFrame({"id": [], "p_active": []})

    total_rows = sum(pq.ParquetFile(f).metadata.num_rows for f in file_paths)
    logging.info(
        f"Starting screening. Found {len(file_paths)} files containing"
        f" {total_rows:,} total molecules."
    )

    results_ids = []
    results_probs = []

    processed_rows = 0
    last_logged_rows = 0

    for file_path in file_paths:
        parquet_file = pq.ParquetFile(file_path)

        for batch in parquet_file.iter_batches(batch_size=batch_size):
            current_batch_size = batch.num_rows
            processed_rows += current_batch_size

            if processed_rows - last_logged_rows >= log_interval:
                percent_complete = (processed_rows / total_rows) * 100
                logging.info(
                    f"Processed {processed_rows:,} / {total_rows:,} rows"
                    f" ({percent_complete:.1f}%)"
                )
                last_logged_rows = processed_rows

            df_batch = pl.from_arrow(batch)
            assert isinstance(df_batch, pl.DataFrame)

            df_clean = df_batch.filter(pl.all_horizontal(pl.col("^.*_valid$")))

            if df_clean.height == 0:
                continue

            x, _, _ = extract_features_and_target(df_clean, feature_cols=feature_cols)

            x_named = (
                x[model.feature_name_]
                if isinstance(x, pl.DataFrame)
                else pl.DataFrame(x, schema=model.feature_name_)
            )

            p_all_classes = np.asarray(model.predict_proba(x_named))
            p_active = p_all_classes[:, 1]

            results_ids.extend(df_clean["id"].to_list())
            results_probs.extend(p_active)

    logging.info(f"Finished! Processed {total_rows:,} / {total_rows:,} rows (100.0%).")
    logging.info(
        f"Successfully generated predictions for {len(results_ids):,} valid molecules."
    )

    df_predictions = pl.DataFrame({"id": results_ids, "p_active": results_probs})

    return df_predictions

In [9]:
df_preds = predict_large_parquet(
    file_pattern="data/features/*.parquet", model=model, feature_cols=feature_cols
)

2026-09-22 21:20:57,395 - INFO - Starting screening. Found 7 files containing 10,200,000 total molecules.
2026-09-22 21:21:03,262 - INFO - Processed 100,000 / 10,200,000 rows (1.0%)
2026-09-22 21:21:09,362 - INFO - Processed 200,000 / 10,200,000 rows (2.0%)
2026-09-22 21:21:15,334 - INFO - Processed 300,000 / 10,200,000 rows (2.9%)
2026-09-22 21:21:21,404 - INFO - Processed 400,000 / 10,200,000 rows (3.9%)
2026-09-22 21:21:27,551 - INFO - Processed 500,000 / 10,200,000 rows (4.9%)
2026-09-22 21:21:33,734 - INFO - Processed 600,000 / 10,200,000 rows (5.9%)
2026-09-22 21:21:39,865 - INFO - Processed 700,000 / 10,200,000 rows (6.9%)
2026-09-22 21:21:46,007 - INFO - Processed 800,000 / 10,200,000 rows (7.8%)
2026-09-22 21:21:52,147 - INFO - Processed 900,000 / 10,200,000 rows (8.8%)
2026-09-22 21:21:58,386 - INFO - Processed 1,000,000 / 10,200,000 rows (9.8%)
2026-09-22 21:22:04,498 - INFO - Processed 1,100,000 / 10,200,000 rows (10.8%)
2026-09-22 21:22:10,616 - INFO - Processed 1,200,000 

In [10]:
k_1_percent = max(1, int(df_preds.height * 0.01))

df_top_1 = df_preds.sort("p_active", descending=True).head(k_1_percent)

print(f"Total screened: {df_preds.height}")
print(f"Top 1% extracted: {df_top_1.height}")

Total screened: 9916668
Top 1% extracted: 99166


In [11]:
def filter_parquet_by_ids(
    source_path: str, ids: list, output_path: str, id_col: str = "id"
) -> None:
    (
        pl.scan_parquet(source_path)
        .filter(pl.col(id_col).is_in(ids))
        .sink_parquet(output_path)
    )

In [14]:
n_samples = min(100_000, df_top_1.height)
sampled_ids = df_top_1["id"].sample(n=n_samples, seed=RANDOM_SEED).to_list()

In [18]:
filter_parquet_by_ids(
    source_path="data/conformers/shard_*",
    ids=sampled_ids,
    output_path="data/conformers/active_round_1.parquet",
)

In [ ]:
features_file = "data/features/shard_0_features.parquet"
active_round_1_features_file = "data/features/active_round_1.parquet"
active_round_1_labels_file = (
    "data/docking_runs/active_round_1_shard_0.1L83.p2rank.1.parquet"
)

ids = pl.read_parquet(active_round_1_labels_file)["id"].to_list()

filter_parquet_by_ids(
    source_path=features_file,
    ids=ids,
    output_path=active_round_1_features_file,
)

In [ ]:
df_active_round_1_clean = load_and_clean_data(
    active_round_1_features_file, active_round_1_labels_file
)
x_active_round_1, y_active_round_1, feature_names = extract_features_and_target(
    df_active_round_1_clean, feature_cols, "affinity_kcal_mol"
)
y_active_round_1_binary = (y_active_round_1 <= active_threshold).astype(int)

In [ ]:
active_threshold

In [ ]:
(
    x_active_round_1_train,
    x_active_round_1_test,
    y_active_round_1_train,
    y_active_round_1_test,
) = train_test_split(
    np.concatenate((x_initial, x_active_round_1)),
    np.concatenate((y_initial_binary, y_active_round_1_binary)),
    train_size=0.8,
    random_state=RANDOM_SEED,
)

In [ ]:
model_active = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=10,
    random_state=RANDOM_SEED,
    deterministic=True,
    force_row_wise=True,
)

model_active.fit(
    x_active_round_1_train, y_active_round_1_train, feature_name=feature_names
)

In [ ]:
y_active_round_1_test, p_active_round_1_test, k_active_round_1_test = (
    prepare_evaluation_data(
        model=model_active,
        x=x_active_round_1_test,
        y=y_active_round_1_test,
        active_threshold=active_threshold,
    )
)

df_metrics = evaluate_enrichment(y_active_round_1_test, p_active_round_1_test)

In [ ]:
y_holdout_binary, p_holdout, k_holdout = prepare_evaluation_data(
    model=model, x=x_holdout, y=y_holdout, active_threshold=active_threshold
)
df_holdout_metrics = evaluate_enrichment(y_holdout_binary, p_holdout)